# Module 3, Part 2: Imitation learning on a real robot arm
## Learning from demonstrations (SO-101 + Hugging Face LeRobot)

Welcome to Part 2. In Part 1, you built the agent-environment loop from scratch: tabular
Q-learning and SARSA on `FrozenLake`, `CliffWalking`, and `CartPole`. Everything there was
*discrete*, *low-dimensional*, and lived inside a *simulator* that handed you a reward.

In this part, the agent is a physical 6-DOF arm sitting in the room. That breaks two
assumptions at once:

1. *Continuous and high-dimensional.* The state is camera pixels plus six joint angles;
   the action is six continuous motor targets. No tidy `Discrete(4)`.
2. *No reward and no simulator.* Nobody is going to write down a reward function for
   "pick up the cube," and we cannot reset the laws of physics 10,000 times to run Q-learning.

So instead of learning from *reward*, we will learn from *demonstration*: a human moves the
arm to show the task, and we fit a policy to that behaviour. This is **imitation learning**.

> **The one idea to carry through the whole session.**
> The simplest form of imitation learning, *behavioural cloning*, is maximum-likelihood
> model fitting on behavioural data. You fit a policy $\pi_\theta(a \mid o)$ to demonstrated
> $(o, a)$ pairs by maximising their likelihood. That is the exact same move you would make
> fitting a Rescorla-Wagner model to a participant's choices in Module 2, only now the "choice" is a
> six-dimensional continuous action and the "stimulus" is an image. Everything you know about
> fitting, recovery, and model comparison still applies.

The session is deliberately short and simple. You will look at real demonstration data, fit a
policy to it on your laptop, and then watch a policy trained exactly this way drive the real
arm at the front of the room. The mini-project picks up from there.

## Setup: Only for instructors (students, skip to Section 0)

**For the session itself**, three things:

1. An assembled, motor-calibrated SO-101 pair, with a webcam, on the instructor laptop.
   Test `lerobot-teleoperate` the day before. That is the only hard dependency.
2. Wifi: students stream a public dataset from the Hugging Face Hub.
3. One pinned LeRobot version, announced to everyone, so 30 installs behave identically.

**For the finale (Section 5) and the mini-project**, one preparation job, done before the school:

1. **Record ~50 teleoperated pick-and-place episodes on our own arm** with `lerobot-record`
   (about an hour), and push them to the Hub. This dataset serves double duty: it trains the
   demo policy below, and it is the dataset the mini-project groups train *their* policies on.
2. **Train ACT on it** on a free Colab GPU with `lerobot-train` (a few hours, unattended), and
   check it does the task. This trained policy is the finale of the session: imitation learning
   actually driving the arm.
3. **Tape the camera down and mark the cube's start positions.** If the camera moves between
   recording and any later deployment, every vision policy dies at once and nobody will be able
   to work out why.

The full pipeline is documented at https://huggingface.co/docs/lerobot/il_robots

**Fallback** if the trained policy is not ready or misbehaves on the day: teleoperate live, then
`lerobot-replay` a recorded episode. Less spectacular, still makes the point.

## 0. Setup and configuration

Everything you run today happens on your own laptop, on the CPU. The arm only appears in the
live demos at the front of the room.

Install once. On Colab, uncomment the `pip` line. Locally, prefer a fresh virtual environment,
as in Part 1.

In [ ]:
# Install (uncomment on Colab or in a fresh environment).
# The version is pinned on purpose. lerobot moves fast, and we would rather thirty laptops
# behave identically today than be surprised by a release that landed last week.
# !pip install "lerobot==0.5.1" matplotlib -q

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Fixed seed, so that the numbers you get are the numbers your neighbour gets.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch", torch.__version__)


In [ ]:
# Configuration: a public dataset of SO-101 demonstrations, streamed from the Hub.
# 50 episodes of pick-and-place, recorded by the LeRobot team. No token needed.
DATASET_REPO_ID = "lerobot/svla_so101_pickplace"


## 1. Why imitation learning?

Before we go anywhere near the robot, it is worth asking how the agent in Part 1 actually learned.

It learned by falling into the lake. Hundreds of times. On `CliffWalking` it walked off the edge
again and again, and the only reason it eventually stopped was that it had accumulated enough
evidence, one catastrophe at a time, that stepping off a cliff has a low value.

Now think about the last thing *you* learned to do. Did you learn it that way?

Animals cannot afford to. A young chimpanzee that worked out which fruits are poisonous by eating
them, or which branch will hold by falling out of the tree, would not be around long enough to
pass the lesson on. And most of the time there is no reward signal to learn from anyway: nobody
hands an infant a scalar for picking up a spoon the right way round. Yet young animals acquire
strikingly complicated behaviour, quickly, and largely by watching somebody who already knows how.

> **Video.** A wild chimpanzee in the Bugoma Forest, Uganda, learning by copying its mother.
> Preprint: https://www.biorxiv.org/content/10.64898/2025.12.01.691567v1
>
> Watch what the young one is actually copying. It is not a reward. It is not a trajectory of joint
> angles either.

The chimpanzee is not a special case. Once you start looking, social learning is everywhere:

- **Nut cracking in the Tai forest.** Young chimpanzees take years to learn to crack nuts with a
  hammer and an anvil, watching their mothers the whole time. Mothers sometimes leave the hammer
  sitting on the anvil for the infant to pick up (Boesch, 1991).
- **Sweet potato washing on Koshima.** In 1953 a young macaque called Imo started washing the sand
  off her sweet potatoes in a stream. The habit spread through the troop along social ties,
  youngsters first, and it is still there (Kawai, 1965).
- **Humpback whales that slap the water before feeding.** Lobtail feeding spread through a
  population of humpbacks over 27 years, and a network analysis showed that it travelled along the
  whales' social connections rather than by genes or by chance (Allen et al., 2013).
- **Meerkats that teach.** Adults bring pups dead scorpions first, then live ones with the sting
  removed, then intact ones. That is a curriculum, and it is tuned to the age of the pup
  (Thornton and McAuliffe, 2006).
- **Children who copy too much.** Show a child a demonstration padded with pointless steps, such as
  tapping the box twice before opening it, and the child will faithfully reproduce the pointless
  steps. Chimpanzees are more sensible and skip them. This is called *over-imitation*, and it is a
  good deal less silly than it looks (Horner and Whiten, 2005; Lyons et al., 2007).

None of this says that reward plays no part in biological learning. It plainly does, and Module 2
was all about how well RL models describe it. The point is that reward is rarely where learning
*starts*. Imitation supplies a strong prior over what is even worth trying, and trial and error
refines it from there. A songbird first memorises its tutor's song and only then practises against
that memory. AlphaGo was pre-trained on human games before it ever played itself. Today's language
models are trained to imitate text first, and tuned with reinforcement learning second.

This part of the tutorial is about that first step, in its most stripped-down machine learning form:
**behavioural cloning**. Show the learner what to do, and fit a policy to it.

### 1.1. Things to think about

1. The infant chimpanzee is not reproducing its mother's joint angles. It is copying something more
   abstract than that. When we clone a robot's behaviour in Section 4, what exactly are we copying,
   and is it the right thing to copy?
2. Could you frame imitation as reinforcement learning after all, with a reward for "matching the
   demonstrator"? What would that reward function be, and would it be any easier to write down than
   a reward for picking up a cube?
3. Over-imitation looks like a bug. Children copy the useless steps; chimpanzees do not. What would
   the equivalent bug be in a robot policy fit to human demonstrations? It has a name, *causal
   confusion*, and chasing it is one of the mini-projects.


## 2. The robot in the room

The SO-101 comes as a pair of arms:

- The *leader* arm: you move it by hand to demonstrate the task.
- The *follower* arm: during recording it mirrors the leader; during evaluation it is driven
  by a policy instead.

That mirroring is the whole trick. By demonstrating, a human produces exactly the
`(observation, action)` pairs a policy needs to learn from. Each recorded frame contains:

| field | meaning | shape |
|---|---|---|
| `observation.state` | the six follower joint angles right now | `(6,)` |
| `observation.images.*` | what the camera(s) see right now | `(3, H, W)` |
| `action` | the six joint targets the leader commanded | `(6,)` |
| `episode_index`, `frame_index`, `timestamp` | bookkeeping | scalars |

Compare this to Part 1. There, the environment handed us a state and a reward, and the agent
chose actions to maximise the reward. Here there is no reward anywhere in the table. A
demonstration is just a time series of "this is what I saw, this is what I did." Learning a
policy means fitting the mapping from observation to action.

### 2.1. Live demo: teleoperation

The instructor now drives the follower with the leader. This is the agent-environment loop
made physical: the human is the policy, the room is the environment. The command, for
reference (it needs the arm on a USB port, so don't run it on your laptop):

```bash
lerobot-teleoperate \
    --robot.type=so101_follower  --robot.port=/dev/ttyACM0 --robot.id=bamb_follower \
    --teleop.type=so101_leader   --teleop.port=/dev/ttyACM1 --teleop.id=bamb_leader \
    --robot.cameras="{ front: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30}}" \
    --display_data=true
```

While the demo runs, watch the operator like a modeller would: how consistent are they from
trial to trial? Where do they slow down? You are looking at a data-generating process, and in
a moment you will model data produced exactly this way.

## 3. Explore the demonstration data

Before fitting anything, look at the behaviour. This was the first lesson of modelling
behavioural data earlier in the school, and it does not stop being true because the participant is a robot
arm. We load a public dataset of 50 pick-and-place demonstrations, recorded through the same
teleoperation loop you just watched.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
# Short on disk? Stream instead of downloading:
#   from lerobot.datasets.streaming_dataset import StreamingLeRobotDataset
#   dataset = StreamingLeRobotDataset(DATASET_REPO_ID, video_backend="pyav")

# video_backend="pyav": decoding the demonstration videos with the default backend needs
# FFmpeg installed system-wide, which most laptops do not have. pyav brings its own copy of
# FFmpeg and ships with lerobot, so this one keyword argument saves everybody an apt-get.
dataset = LeRobotDataset(DATASET_REPO_ID, video_backend="pyav")

print("episodes:     ", dataset.num_episodes)
print("total frames: ", dataset.num_frames)
print("control fps:  ", dataset.fps)
print("features:")
for name, feat in dataset.features.items():
    print(f"    {name:35s} {feat.get('shape', None)}")


In [ ]:
# One frame is one (observation, action) pair, i.e. one training example.
sample = dataset[0]
for k, v in sample.items():
    print(f"{k:35s} {tuple(v.shape) if torch.is_tensor(v) else v}")

# Detect the camera key(s) and dimensions from the dataset itself.
IMAGE_KEYS = [k for k in dataset.features if k.startswith("observation.images")]
CAM        = IMAGE_KEYS[0]
STATE_DIM  = dataset.features["observation.state"]["shape"][0]
ACTION_DIM = dataset.features["action"]["shape"][0]
print("\ncameras:", IMAGE_KEYS, "| state dim:", STATE_DIM, "| action dim:", ACTION_DIM)

### 3.1. What does the robot see, and what does it do?

In [ ]:
# What the camera saw, at four points across the dataset.
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, idx in zip(axes, np.linspace(0, len(dataset) - 1, 4).astype(int)):
    ax.imshow(dataset[int(idx)][CAM].permute(1, 2, 0).numpy())
    ax.set_title(f"frame {idx}")
    ax.axis("off")
fig.suptitle(f"The policy's visual input ({CAM})")
plt.tight_layout()
plt.show()

In [ ]:
# The behaviour itself. Tabular signals live in Parquet, so we can pull them all as arrays
# without decoding a single video frame. Fast even on a laptop.
tab         = dataset.hf_dataset
states_all  = np.array(tab["observation.state"], dtype=np.float32)   # (N, STATE_DIM)
actions_all = np.array(tab["action"],            dtype=np.float32)   # (N, ACTION_DIM)
episodes    = np.array(tab["episode_index"])                          # (N,)

ep = 0
m = episodes == ep
t = np.arange(m.sum()) / dataset.fps
fig, ax = plt.subplots(figsize=(9, 4))
for j in range(ACTION_DIM):
    ax.plot(t, actions_all[m, j], label=f"joint {j}")
ax.set_xlabel("time (s)")
ax.set_ylabel("commanded action")
ax.legend(ncol=3, fontsize=8)
ax.set_title(f"Episode {ep}: a demonstration is a 6-D trajectory through time")
plt.show()

### 3.2. TO DO: How variable are the demonstrations?

In Module 2 you cared about individual differences between participants. The same idea shows up
here as variability between demonstrations: even for "the same" task, no two episodes are
identical, and that variability is exactly what a policy will have to cope with.

Overlay one joint (start with joint 0) across the first 8 episodes. Time-align them by
starting each episode at t = 0.

In [ ]:
joint = 0
plt.figure(figsize=(9, 4))
for ep in range(min(8, dataset.num_episodes)):
    m = episodes == ep
    # TO DO: compute the time axis for this episode and plot actions_all[m, joint]
    # t = ...
    # plt.plot(...)
    pass
plt.xlabel("time (s)")
plt.ylabel(f"action, joint {joint}")
plt.title("Between-demonstration variability")
plt.show()

### 3.3. Questions

1. Where do the episodes agree most, and where do they diverge: during the initial reach, or
   around the moment of grasping and placing? Why would that be?
2. If two demonstrations start from the same joint angles but the object is in different
   places, can any policy that only knows the joint angles produce both behaviours? Keep this
   question in mind; it is the punchline of the next section.

## 4. Behavioural cloning: fit your first robot policy

We fit our first policy, and we make it deliberately naive: a small neural network that maps
the current joint angles to the commanded action, ignoring the camera entirely. It only feels
its own body, so we will call it the *blind clone*.

$$\pi_\theta(a \mid s) = \mathcal{N}\big(f_\theta(s),\, \sigma^2 I\big)$$

Training this network to minimise the mean-squared error between its output and the
demonstrated action is exactly maximising the Gaussian log-likelihood of the demonstrations.
So the cell below is doing the same thing you did when fitting Rescorla-Wagner by maximum
likelihood, with two differences: the model is a neural network, and the "choices" are
six-dimensional and continuous.

One rule for the split: split by episode, never by frame. Consecutive frames within an
episode are nearly identical, so a frame-wise split would leak the test set into training and
flatter the model. Same leakage logic as splitting trials within a session when you fit behavioural models.

In [ ]:
# Split by episode: hold out the last 20% of episodes.
ep_ids   = np.unique(episodes)
n_train  = int(0.8 * len(ep_ids))
train_ep = set(ep_ids[:n_train])
test_ep  = set(ep_ids[n_train:])
is_train = np.array([e in train_ep for e in episodes])

Xtr = torch.tensor(states_all[is_train]);  Ytr = torch.tensor(actions_all[is_train])
print(f"train frames: {is_train.sum()} | test frames: {(~is_train).sum()}")

# Standardise inputs and outputs. LeRobot's own training pipelines do the equivalent.
x_mean, x_std = Xtr.mean(0), Xtr.std(0) + 1e-6
y_mean, y_std = Ytr.mean(0), Ytr.std(0) + 1e-6

In [ ]:
class BlindClone(nn.Module):
    '''MLP from joint angles to actions. No vision.'''
    def __init__(self, d_in, d_out, hidden=128):
        super().__init__()
        # Buffers, not plain globals: they travel with the model when you save it. You will
        # thank us when you have to ship a checkpoint to the arm.
        self.register_buffer("x_mean", x_mean); self.register_buffer("x_std", x_std)
        self.register_buffer("y_mean", y_mean); self.register_buffer("y_std", y_std)
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, d_out),
        )
    def forward(self, s):
        return self.net((s - self.x_mean) / self.x_std) * self.y_std + self.y_mean

# Re-seed here too: you will want to come back and change things (the hidden size, the
# learning rate, the number of epochs), and re-running this one cell should then differ
# only by what you changed, not by a fresh roll of the initial weights.
torch.manual_seed(SEED)

blind = BlindClone(STATE_DIM, ACTION_DIM)
opt   = torch.optim.Adam(blind.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()   # MSE is the Gaussian negative log-likelihood
loader  = DataLoader(TensorDataset(Xtr, Ytr), batch_size=256, shuffle=True)

losses = []
for epoch in range(40):
    epoch_loss = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(blind(xb), yb)
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * len(xb)
    losses.append(epoch_loss / len(loader.dataset))   # mean over the epoch, not the last batch

plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("Fitting the clone = maximising the likelihood of the demonstrations")
plt.show()

In [ ]:
# Evaluate on a held-out episode: given what the robot observed at each moment, how close
# are the clone's predicted actions to what the human actually did? This is the policy
# analogue of comparing model-predicted choice probabilities to held-out real choices.
test_ep_id = sorted(test_ep)[0]
m = episodes == test_ep_id

with torch.no_grad():
    pred_blind = blind(torch.tensor(states_all[m])).numpy()
true = actions_all[m]
print(f"held-out action error (blind clone): {np.abs(pred_blind - true).mean():.4f}")

t = np.arange(m.sum()) / dataset.fps
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
for j, ax in enumerate(axes.flat):
    ax.plot(t, true[:, j], label="demonstrated", lw=2)
    ax.plot(t, pred_blind[:, j], label="blind clone", ls="--")
    ax.set_title(f"joint {j}")
axes.flat[0].legend()
fig.suptitle("Blind clone vs demonstration, held-out episode")
plt.tight_layout()
plt.show()

### 4.1. What did we just do, and what didn't we do?

That is behavioural cloning, end to end: demonstrations in, maximum-likelihood fit, a policy
out. It is the same workflow you used on human choice data in Module 2, and everything you know
from there — train/test splits, baselines, model comparison — carries over unchanged.

Two honest caveats before anyone gets carried away, and they are both doors into the mini-project:

1. **Our clone is blind.** Its only input is the six joint angles; it never sees the camera.
   When the cube starts somewhere new, the same joint angles demand a different action, and no
   amount of extra layers can help a network whose input does not contain the information. What
   an agent can learn is bounded by what it represents — Module 2's model-space lesson, now in
   observation space. The fix is to change the *input*, not the model: give the policy the camera.

2. **Predicting a demonstration well is not the same as doing the task well.** Our score asks,
   frame by frame, "what would you have done here?", always handing the policy the true state,
   so its mistakes never come home to roost. A deployed policy has no such luxury: its own
   actions determine what it sees next, and small errors can drift it into states no demonstrator
   ever visited. The only real test of a policy is letting it drive the arm — which is exactly
   what happens next, and what the best mini-project policies will get to do.

### Things to think about

1. Section 1's over-imitating children copied the causally useless steps of a demonstration.
   What might a cloned policy copy from these demonstrations that is causally useless for the
   task? (This has a name, *causal confusion*, and the mini-project description has more.)
2. If two demonstrations start from the same joint angles but the cube is in different places,
   what is the best a blind policy can possibly do?

## 5. The finale: imitation learning drives the real arm

Everything you just did — record demonstrations, fit a policy by maximum likelihood — is the
*entire* recipe used on real robots. The only differences between your two-minute clone and a
deployable policy are the ones from Section 4.1: real policies take the camera as input, and
they predict a short *chunk* of future actions instead of one action at a time, so that small
errors do not compound frame by frame. The standard such policy for this arm is
[ACT](https://arxiv.org/abs/2304.13705), the Action Chunking Transformer.

We prepared one. Before the school, we teleoperated ~50 pick-and-place demonstrations **on this
arm**, and trained ACT on them overnight on a free Colab GPU — no reward function, no simulator,
just demonstrations, exactly as in this notebook. The instructor now runs it:

```bash
lerobot-record \
    --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=bamb_follower \
    --robot.cameras="{ front: {type: opencv, index_or_path: 0, width: 640, height: 480, fps: 30}}" \
    --dataset.repo_id=<instructor>/eval_act_so101 --dataset.single_task="pick and place" \
    --policy.path=<instructor>/act_so101_pickplace
```

No human is touching the leader arm. The policy is seeing camera images and joint angles, thirty
times a second, and answering with motor targets it learned from an hour of demonstrations.

One detail matters enough to say out loud: **the policy works because it was trained on data
from this arm, this table, this camera.** The public dataset you fit in Section 4 was recorded
on somebody else's setup; a policy trained on it, however accurate, would reach for where *their*
cube used to be. (If the instructor replays one of its episodes through our arm with
`lerobot-replay`, you can watch it grasp thin air.) Demonstrations do not transfer across scenes;
this is why the mini-project trains on *our* dataset. Its Hub id is in
[`mini_projects.md`](../mini_projects.md).

## 6. The mini-project

You have now done the whole loop in miniature: demonstration data in, a policy out, fit by
maximum likelihood. The mini-project is to make that policy good enough to trust on the real arm.

> **Train a policy on our demonstration dataset. The best few policies in the class get deployed
> on the real arm on the last day.**

Everything you need — the dataset, the two upgrades that turn a blind clone into a real policy,
and how a policy gets chosen for deployment — is in [`mini_projects.md`](../mini_projects.md).

## References and further material

### The robot, the data, and the algorithms

- LeRobot documentation: https://huggingface.co/docs/lerobot
- Imitation learning on real robots, the full record-train-deploy walkthrough:
  https://huggingface.co/docs/lerobot/il_robots
- The public dataset used in Section 4: https://huggingface.co/datasets/lerobot/svla_so101_pickplace
- ACT, the Action Chunking Transformer (Zhao et al., 2023): https://arxiv.org/abs/2304.13705
- What makes a good dataset: https://huggingface.co/blog/lerobot-datasets
- Causal confusion in imitation learning (de Haan et al., 2019): https://arxiv.org/abs/1905.11979
- LeRobot community Discord, live support during the school:
  https://discord.com/invite/s3KuuzsPFb

### Learning by watching

The chimpanzee video and a reading list for Section 1 are collected in the
[part 2 README](./README.md).

- The chimpanzees of the Bugoma Forest, Uganda: https://www.biorxiv.org/content/10.64898/2025.12.01.691567v1
- Horner and Whiten (2005), causal knowledge and imitation in chimpanzees and children:
  https://doi.org/10.1007/s10071-004-0239-6
- Allen et al. (2013), network-based diffusion analysis reveals cultural transmission of lobtail
  feeding in humpback whales: https://doi.org/10.1126/science.1231976